# Dora Maar — Artnet Image Scraper

Scrapes all Dora Maar artworks from Artnet using a visible Firefox browser, downloads every thumbnail image, and saves a metadata CSV with file paths.

In [1]:
import os, ssl, certifi
os.environ["SSL_CERT_FILE"]      = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
ssl._create_default_https_context = ssl.create_default_context

import re
import time
import pandas as pd
import browser_cookie3
from pathlib import Path
from bs4 import BeautifulSoup
from curl_cffi import requests as curl_requests

IMAGES_DIR = Path("images")
IMAGES_DIR.mkdir(exist_ok=True)

print("Setup complete. Images will be saved to:", IMAGES_DIR.resolve())

Setup complete. Images will be saved to: /Users/pietro/Desktop/Maar/images


---
## Step 1 — Fetch all 64 Artnet pages

**Before running this cell:**  
1. Open Firefox manually and go to `https://www.artnet.com/artists/dora-maar/`  
2. Solve any CAPTCHA until artworks are visible  
3. Come back here and run the cell — cookies are read directly from your Firefox profile

In [ ]:
TOTAL_PAGES = 64
BASE_URL    = "https://www.artnet.com/artists/dora-maar/{}"

# --- Read cookies straight from your Firefox profile (no Selenium needed) ---
ff_cookies = browser_cookie3.firefox(domain_name=".artnet.com")
cookie_list = list(ff_cookies)
print(f"Read {len(cookie_list)} artnet.com cookies from Firefox profile:\n")
for c in cookie_list:
    print(f"  {c.name:40s}  {c.value[:60]}")

incapsula = [c for c in cookie_list if "incap" in c.name.lower() or "visid" in c.name.lower()]
print(f"\nIncapsula cookies found: {len(incapsula)}")
for c in incapsula:
    print(f"  {c.name} = {c.value[:80]}")

if not incapsula:
    print("\n⚠  No Incapsula cookies found.")
    print("   Make sure you opened artnet.com in Firefox and solved any CAPTCHA before running this cell.")

# --- curl_cffi session: Firefox TLS fingerprint + your real browser cookies ---
session = curl_requests.Session(impersonate="firefox135")
for c in cookie_list:
    session.cookies.set(c.name, c.value, domain=c.domain)

print(f"\nSession ready. Fetching {TOTAL_PAGES} pages...\n")

# --- Fetch all 64 pages ---
html_pages = {}

for page in range(1, TOTAL_PAGES + 1):
    url = BASE_URL.format(page)
    try:
        resp = session.get(url, timeout=15)
        size = len(resp.text)
        blocked = "Incapsula" in resp.text or size < 5_000
        status = "BLOCKED ⚠" if blocked else "ok"
        print(f"  Page {page:2d}/{TOTAL_PAGES}  {resp.status_code}  {size:>8,} chars  [{status}]")
        html_pages[page] = "" if blocked else resp.text
    except Exception as e:
        print(f"  Page {page:2d}/{TOTAL_PAGES}  ERROR: {e}")
        html_pages[page] = ""
    if page < TOTAL_PAGES:
        time.sleep(5)

ok = sum(1 for h in html_pages.values() if h)
print(f"\nDone. {ok}/{TOTAL_PAGES} pages fetched successfully.")


In [4]:
from artnet_html_parser import save_pages
# --- Save HTML to disk so Step 2 can be re-run without re-fetching ---
save_pages(html_pages, "artnet_html")

Saved 64 HTML files to /Users/pietro/Desktop/Maar/artnet_html


---
## Step 2 — Parse artworks & save catalog CSV

Loads HTML from `artnet_html/`, parses every artwork block, and saves `artnet_catalog_raw.csv`.  
Columns: `page, title, year, gallery_name, gallery_url, price, img_url, detail_url`

In [ ]:
html_pages[1]

'<html style="height:100%"><head><META NAME="ROBOTS" CONTENT="NOINDEX, NOFOLLOW"><meta name="format-detection" content="telephone=no"><meta name="viewport" content="initial-scale=1.0"><meta http-equiv="X-UA-Compatible" content="IE=edge,chrome=1"><script type="text/javascript" src="/_Incapsula_Resource?SWJIYLWA=719d34d31c8e3a6e6fffd425f7e032f3"></script><script src="/nly-What-neuer-of-my-But-of-Rosse-a-Say-thinke-v" async></script><script type="text/javascript">if (sessionStorage) { sessionStorage.setItem(\'distil_referrer\', document.referrer); }</script></head><body style="margin:0px;height:100%"><iframe id="main-iframe" src="/_Incapsula_Resource?SWUDNSAI=31&xinfo=3-10314219-0%202NNN%20RT%281776914947886%2045%29%20q%280%20-1%20-1%200%29%20r%280%20-1%29%20B12%2811%2c10599801%2c0%29%20U18&incident_id=358000830061427265-46692131391542467&edet=12&cinfo=0b000000&rpinfo=0&cts=pFevviYeYv9e3MK6f6i8CY5ZRMwHmvSdN5S%2bsimk5DXgNiUbjEQ8igWgDGyISaX8&cip=47.230.133.129&mth=GET" frameborder=0 width="

In [5]:
from artnet_html_parser import parse_pages, load_pages

CATALOG_RAW = Path("artnet_catalog_raw.csv")

html_pages = load_pages("artnet_html")
print(f"Loaded {len(html_pages)} HTML files from artnet_html/")

df = parse_pages(html_pages)
print(f"Parsed {len(df)} unique artworks")

df.to_csv(CATALOG_RAW, index=False)
print(f"Saved to {CATALOG_RAW.resolve()}")
df.head(10)

Loaded 64 HTML files from artnet_html/
Parsed 1519 unique artworks
Saved to /Users/pietro/Desktop/Maar/artnet_catalog_raw.csv


,page,title,year,img_url,detail_url
0,1,Body studies,1998,https://www.artnet.com/WebServices/images/ll30...,https://www.artnet.com/artists/dora-maar/body-...
1,1,Picasso,1936,https://www.artnet.com/WebServices/images/ll29...,https://www.artnet.com/artists/dora-maar/picas...
2,1,"Study of Light on Water, (Études de Lumière...",1935,https://www.artnet.com/WebServices/images/ll28...,https://www.artnet.com/artists/dora-maar/study...
3,1,"Booksellers [Couple], (Bouquinistes...",1935,https://www.artnet.com/WebServices/images/ll28...,https://www.artnet.com/artists/dora-maar/books...
4,1,"Rooves [Notre-Dame in the Distance], Paris,...",1935,https://www.artnet.com/WebServices/images/ll28...,https://www.artnet.com/artists/dora-maar/roove...
5,1,"Window Display (Hairdresser's), Paris...",1935,https://www.artnet.com/WebServices/images/ll28...,https://www.artnet.com/artists/dora-maar/windo...
6,1,Woman with a Fur Collar in Front of a...,1935,https://www.artnet.com/WebServices/images/ll28...,https://www.artnet.com/artists/dora-maar/woman...
7,1,"Londoners Sitting [Park], London (Londoniens",1934,https://www.artnet.com/WebServices/images/ll28...,https://www.artnet.com/artists/dora-maar/londo...
8,1,"Lifeboat, Calais",1935,https://www.artnet.com/WebServices/images/ll28...,https://www.artnet.com/artists/dora-maar/lifeb...
9,1,Woman in the Streets of Paris [Back View July...,,https://www.artnet.com/WebServices/images/ll28...,https://www.artnet.com/artists/dora-maar/woman...


---
## Step 3 — Download images & save metadata CSV

Images saved to `images/` as `{index:04d}_{slug}.jpg`.  
Progress is written to `artnet_catalog.csv` every 50 downloads so the run can be resumed.

In [6]:
CSV_PATH = Path("artnet_catalog.csv")

# Resume from existing CSV if present
if CSV_PATH.exists():
    df_done = pd.read_csv(CSV_PATH)
    done_urls = set(df_done["img_url"])
    print(f"Resuming — {len(df_done)} already downloaded.")
else:
    df_done = pd.DataFrame()
    done_urls = set()

records = []
img_session = curl_requests.Session(impersonate="firefox135")

for idx, row in df.iterrows():
    if row["img_url"] in done_urls:
        continue

    slug = re.sub(r"[^a-z0-9]+", "-", row["img_url"].split("/")[-1].lower())[:80]
    filename = f"{idx:04d}_{slug}"
    if not filename.endswith(".jpg"):
        filename += ".jpg"
    local_path = IMAGES_DIR / filename

    try:
        resp = img_session.get(row["img_url"], timeout=15)
        resp.raise_for_status()
        local_path.write_bytes(resp.content)
        status = "ok"
    except Exception as e:
        local_path = Path("")
        status = f"error: {e}"

    records.append({
        **row.to_dict(),
        "local_path": str(local_path),
        "status":     status,
    })

    if len(records) % 50 == 0:
        batch = pd.concat([df_done, pd.DataFrame(records)], ignore_index=True)
        batch.to_csv(CSV_PATH, index=False)
        print(f"  Checkpoint: {len(batch)} rows saved.")

    time.sleep(0.15)

df_catalog = pd.concat([df_done, pd.DataFrame(records)], ignore_index=True)
df_catalog.to_csv(CSV_PATH, index=False)

ok = (df_catalog["status"] == "ok").sum()
print(f"\nDone. {ok} / {len(df_catalog)} images downloaded.")
print(f"Catalog saved to: {CSV_PATH.resolve()}")
df_catalog.head(10)

  Checkpoint: 50 rows saved.
  Checkpoint: 100 rows saved.
  Checkpoint: 150 rows saved.
  Checkpoint: 200 rows saved.
  Checkpoint: 250 rows saved.
  Checkpoint: 300 rows saved.
  Checkpoint: 350 rows saved.
  Checkpoint: 400 rows saved.
  Checkpoint: 450 rows saved.
  Checkpoint: 500 rows saved.
  Checkpoint: 550 rows saved.
  Checkpoint: 600 rows saved.
  Checkpoint: 650 rows saved.
  Checkpoint: 700 rows saved.
  Checkpoint: 750 rows saved.
  Checkpoint: 800 rows saved.
  Checkpoint: 850 rows saved.
  Checkpoint: 900 rows saved.
  Checkpoint: 950 rows saved.
  Checkpoint: 1000 rows saved.
  Checkpoint: 1050 rows saved.
  Checkpoint: 1100 rows saved.
  Checkpoint: 1150 rows saved.
  Checkpoint: 1200 rows saved.
  Checkpoint: 1250 rows saved.
  Checkpoint: 1300 rows saved.
  Checkpoint: 1350 rows saved.
  Checkpoint: 1400 rows saved.
  Checkpoint: 1450 rows saved.
  Checkpoint: 1500 rows saved.

Done. 1512 / 1519 images downloaded.
Catalog saved to: /Users/pietro/Desktop/Maar/artnet_

,page,title,year,img_url,detail_url,local_path,status
0,1,Body studies,1998,https://www.artnet.com/WebServices/images/ll30...,https://www.artnet.com/artists/dora-maar/body-...,images/0000_dora-maar-body-studies-jpg.jpg,ok
1,1,Picasso,1936,https://www.artnet.com/WebServices/images/ll29...,https://www.artnet.com/artists/dora-maar/picas...,images/0001_dora-maar-picasso-jpg.jpg,ok
2,1,"Study of Light on Water, (Études de Lumière...",1935,https://www.artnet.com/WebServices/images/ll28...,https://www.artnet.com/artists/dora-maar/study...,images/0002_dora-maar-study-of-light-on-water-...,ok
3,1,"Booksellers [Couple], (Bouquinistes...",1935,https://www.artnet.com/WebServices/images/ll28...,https://www.artnet.com/artists/dora-maar/books...,images/0003_dora-maar-booksellers-couple-bouqu...,ok
4,1,"Rooves [Notre-Dame in the Distance], Paris,...",1935,https://www.artnet.com/WebServices/images/ll28...,https://www.artnet.com/artists/dora-maar/roove...,images/0004_dora-maar-rooves-notre-dame-in-the...,ok
5,1,"Window Display (Hairdresser's), Paris...",1935,https://www.artnet.com/WebServices/images/ll28...,https://www.artnet.com/artists/dora-maar/windo...,images/0005_dora-maar-window-display-hairdress...,ok
6,1,Woman with a Fur Collar in Front of a...,1935,https://www.artnet.com/WebServices/images/ll28...,https://www.artnet.com/artists/dora-maar/woman...,images/0006_dora-maar-woman-with-a-fur-collar-...,ok
7,1,"Londoners Sitting [Park], London (Londoniens",1934,https://www.artnet.com/WebServices/images/ll28...,https://www.artnet.com/artists/dora-maar/londo...,images/0007_dora-maar-londoners-sitting-park-l...,ok
8,1,"Lifeboat, Calais",1935,https://www.artnet.com/WebServices/images/ll28...,https://www.artnet.com/artists/dora-maar/lifeb...,images/0008_dora-maar-lifeboat-calais-jpg.jpg,ok
9,1,Woman in the Streets of Paris [Back View July...,,https://www.artnet.com/WebServices/images/ll28...,https://www.artnet.com/artists/dora-maar/woman...,images/0009_dora-maar-woman-in-the-streets-of-...,ok
